In [14]:
from pathlib import Path
import pandas as pd
import numpy as np
import json

BASE_DIR = Path(
    r"D:\DATA SCIENCE COURSE\ML\ML_Projects\WasteWise-AI"
)

RAW_DIR = BASE_DIR / "datasets" / "raw"
OUT_DIR = BASE_DIR / "datasets" / "processed"
REPORT_DIR = BASE_DIR / "datasets" / "reports"

OUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("Raw folder exists:", RAW_DIR.exists())


Raw folder exists: True


In [15]:
file_path = RAW_DIR / "discount_scenarios_simulated.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

Dataset loaded successfully!
Shape: (10000, 8)


In [16]:
display(df.head())

print("Columns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nExact duplicate rows:", df.duplicated().sum())

,batch_id,product_id,days_to_expiry,quantity_remaining,discount_pct,estimated_units_sold_simulated,estimated_revenue_simulated,estimated_gross_margin_simulated
0,5001,1006,5,72,0,72,3240.0,1224.0
1,5001,1006,5,72,5,72,3078.0,1062.0
2,5001,1006,5,72,10,72,2916.0,900.0
3,5001,1006,5,72,15,72,2754.0,738.0
4,5001,1006,5,72,20,72,2592.0,576.0


Columns:
['batch_id', 'product_id', 'days_to_expiry', 'quantity_remaining', 'discount_pct', 'estimated_units_sold_simulated', 'estimated_revenue_simulated', 'estimated_gross_margin_simulated']

Data types:
batch_id                              int64
product_id                            int64
days_to_expiry                        int64
quantity_remaining                    int64
discount_pct                          int64
estimated_units_sold_simulated        int64
estimated_revenue_simulated         float64
estimated_gross_margin_simulated    float64
dtype: object

Missing values:
batch_id                            0
product_id                          0
days_to_expiry                      0
quantity_remaining                  0
discount_pct                        0
estimated_units_sold_simulated      0
estimated_revenue_simulated         0
estimated_gross_margin_simulated    0
dtype: int64

Exact duplicate rows: 0


In [17]:
print("Unique discount percentages:")
print(sorted(df["discount_pct"].dropna().unique()))

print("\nRows per discount percentage:")
print(df["discount_pct"].value_counts(dropna=False).sort_index())

Unique discount percentages:
[np.int64(0), np.int64(5), np.int64(10), np.int64(15), np.int64(20)]

Rows per discount percentage:
discount_pct
0     2000
5     2000
10    2000
15    2000
20    2000
Name: count, dtype: int64


In [18]:
print("Columns:", df.columns.tolist())

for column in ["batch_id", "product_id"]:
    if column in df.columns:
        df[column] = df[column].astype("string").str.strip()

display(df.head())

Columns: ['batch_id', 'product_id', 'days_to_expiry', 'quantity_remaining', 'discount_pct', 'estimated_units_sold_simulated', 'estimated_revenue_simulated', 'estimated_gross_margin_simulated']


,batch_id,product_id,days_to_expiry,quantity_remaining,discount_pct,estimated_units_sold_simulated,estimated_revenue_simulated,estimated_gross_margin_simulated
0,5001,1006,5,72,0,72,3240.0,1224.0
1,5001,1006,5,72,5,72,3078.0,1062.0
2,5001,1006,5,72,10,72,2916.0,900.0
3,5001,1006,5,72,15,72,2754.0,738.0
4,5001,1006,5,72,20,72,2592.0,576.0


In [19]:
numeric_columns = [
    column
    for column in [
        "discount_pct",
        "unit_price",
        "unit_cost",
        "expected_demand_before_expiry",
        "predicted_waste_units",
        "waste_units_predicted",
        "expected_waste_units",
        "revenue",
        "profit"
    ]
    if column in df.columns
]

for column in numeric_columns:
    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )

print("Numeric columns converted:", numeric_columns)
print(df[numeric_columns].dtypes)

Numeric columns converted: ['discount_pct']
discount_pct    int64
dtype: object


In [20]:
print("Missing values:")
print(df.isnull().sum())

print("\nExact duplicate rows:", df.duplicated().sum())

Missing values:
batch_id                            0
product_id                          0
days_to_expiry                      0
quantity_remaining                  0
discount_pct                        0
estimated_units_sold_simulated      0
estimated_revenue_simulated         0
estimated_gross_margin_simulated    0
dtype: int64

Exact duplicate rows: 0


In [21]:
invalid_discount_mask = (
    df["discount_pct"].isna()
    | ~df["discount_pct"].isin([0, 5, 10, 15, 20])
)

print("Rows with invalid discount percentages:",
      invalid_discount_mask.sum())

Rows with invalid discount percentages: 0


In [22]:
rejected_df = df[invalid_discount_mask].copy()

clean_df = df[~invalid_discount_mask].copy()

print("Original rows:", len(df))
print("Rejected rows:", len(rejected_df))

print("Rows remaining:", len(clean_df))

Original rows: 10000
Rejected rows: 0
Rows remaining: 10000


In [23]:
before = len(clean_df)

clean_df = clean_df.drop_duplicates().copy()

print("Exact duplicates removed:", before - len(clean_df))
print("Rows after duplicate removal:", len(clean_df))

Exact duplicates removed: 0
Rows after duplicate removal: 10000


In [24]:
if "batch_id" in clean_df.columns:
    scenario_counts = (
        clean_df.groupby("batch_id")["discount_pct"]
        .nunique()
    )

    print("Batches with five discount scenarios:",
          (scenario_counts == 5).sum())

    print("Batches with an unexpected scenario count:",
          (scenario_counts != 5).sum())

Batches with five discount scenarios: 2000
Batches with an unexpected scenario count: 0


In [25]:
clean_path = OUT_DIR / "discount_scenarios_simulated_clean.csv"
rejected_path = REPORT_DIR / "discount_scenarios_rejected.csv"
report_path = REPORT_DIR / "discount_scenarios_report.json"

clean_df.to_csv(clean_path, index=False)
rejected_df.to_csv(rejected_path, index=False)

report = {
    "original_rows": int(len(df)),
    "clean_rows": int(len(clean_df)),
    "rejected_rows": int(len(rejected_df)),
    "duplicate_rows_removed": int(
        len(df) - len(rejected_df) - len(clean_df)
    ),
    "columns": clean_df.columns.tolist()
}

with open(report_path, "w", encoding="utf-8") as file:
    json.dump(report, file, indent=4)

print("Clean dataset:", clean_path)
print("Rejected rows:", rejected_path)
print("Cleaning report:", report_path)

Clean dataset: D:\DATA SCIENCE COURSE\ML\ML_Projects\WasteWise-AI\datasets\processed\discount_scenarios_simulated_clean.csv
Rejected rows: D:\DATA SCIENCE COURSE\ML\ML_Projects\WasteWise-AI\datasets\reports\discount_scenarios_rejected.csv
Cleaning report: D:\DATA SCIENCE COURSE\ML\ML_Projects\WasteWise-AI\datasets\reports\discount_scenarios_report.json


In [26]:
print("Final shape:", clean_df.shape)
print("Missing values:", clean_df.isnull().sum().sum())
print("Exact duplicate rows:", clean_df.duplicated().sum())

display(clean_df.head())

Final shape: (10000, 8)
Missing values: 0
Exact duplicate rows: 0


,batch_id,product_id,days_to_expiry,quantity_remaining,discount_pct,estimated_units_sold_simulated,estimated_revenue_simulated,estimated_gross_margin_simulated
0,5001,1006,5,72,0,72,3240.0,1224.0
1,5001,1006,5,72,5,72,3078.0,1062.0
2,5001,1006,5,72,10,72,2916.0,900.0
3,5001,1006,5,72,15,72,2754.0,738.0
4,5001,1006,5,72,20,72,2592.0,576.0
